In [14]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score , classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import mlflow
from pathlib import Path


import matplotlib.pyplot as plt
import seaborn as sns

# Load datasets

In [15]:
TRAIN_PATH = "./../data/processed/train.csv"
VALIDATION_PATH = "./../data/processed/validation.csv"
TEST_PATH = "./../data/processed/test.csv"

In [16]:
train_df= pd.read_csv(TRAIN_PATH)
validation_df= pd.read_csv(VALIDATION_PATH)
test_df= pd.read_csv(TEST_PATH)

X_train = train_df.drop(columns=["target"])
y_train = train_df["target"]

X_validation = validation_df.drop(columns=["target"])
y_validation = validation_df["target"]

X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

del train_df, validation_df, test_df

In [17]:
print("Training set shape:", X_train.shape)
print("Validation set shape:", X_validation.shape)
print("Test set shape:", X_test.shape)

Training set shape: (17743, 27)
Validation set shape: (3132, 27)
Test set shape: (2320, 27)


In [18]:
print(X_train.isna().sum().sum())
print(X_validation.isna().sum().sum())
print(X_test.isna().sum().sum())

0
0
0


In [19]:
print(y_train.isna().sum().sum())
print(y_validation.isna().sum().sum())
print(y_test.isna().sum().sum())

0
0
0


# MLFLOW

In [20]:
EXPERIMENT_NAME="house_price_prediction_experiment"
TRAIN_CONFUSION_MATRIX_PATH = './../reports/train_confusion_matrix.png'
VALIDATION_CONFUSION_MATRIX_PATH = './../reports/validation_confusion_matrix.png'
TEST_CONFUSION_MATRIX_PATH = './../reports/test_confusion_matrix.png'

In [21]:
base_dir = Path("./../models").resolve()
mlflow.set_tracking_uri(f"sqlite:///{base_dir / 'mlflow.db'}")
mlflow.set_experiment(EXPERIMENT_NAME)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment = mlflow.set_experiment(EXPERIMENT_NAME)

experiment_id = experiment.experiment_id

print("EXPERIMENT INFO:")
print(f"Name: {experiment.name}")
print(f"ID: {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")
print(f"Tags: {experiment.tags}")
print(f"Lifecycle Stage: {experiment.lifecycle_stage}")
print(f"Creation timestamp: {experiment.creation_time}")

EXPERIMENT INFO:
Name: house_price_prediction_experiment
ID: 1
Artifact Location: file:d:/fourth_year/second_sem/applied_data_science/DS_Project/notebooks/mlruns/1
Tags: {}
Lifecycle Stage: active
Creation timestamp: 1777678380054


# classification evaluation

In [22]:
def save_confusion_matrix(cm, filename):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d") 
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.savefig(filename)
    plt.close()

In [23]:
def evaluate_model(y_pred, y_true):
    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    return acc, report, cm

# Model training

### logistic regression

In [24]:
with mlflow.start_run(run_name='logistic_regression_v1', experiment_id=experiment_id):
    params = {
        'penalty': 'l2',         
        'C': 1.0,                
        'solver': 'lbfgs',       
        'max_iter': 10000,        
        'random_state': 42
    }

    logreg = LogisticRegression(**params)
    logreg.fit(X_train, y_train)

    y_train_pred = logreg.predict(X_train)
    y_validation_pred = logreg.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    print("Training Accuracy:", train_acc)
    print("Validation Accuracy:", val_acc)
    print("Training Classification Report:\n", train_report)
    print("Validation Classification Report:\n", val_report)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH)
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.log_params(params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc
    })
    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')
    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(logreg, 'logistic_regression_model')

d:\anaconda\envs\genai-env\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 10000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=10000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training Accuracy: 0.5327734881361664
Validation Accuracy: 0.5335249042145593
Training Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.61      0.60      5915
           1       0.44      0.34      0.39      5921
           2       0.54      0.64      0.59      5907

    accuracy                           0.53     17743
   macro avg       0.52      0.53      0.52     17743
weighted avg       0.52      0.53      0.52     17743

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.61      0.60      1044
           1       0.44      0.32      0.37      1045
           2       0.54      0.67      0.60      1043

    accuracy                           0.53      3132
   macro avg       0.52      0.53      0.52      3132
weighted avg       0.52      0.53      0.52      3132



2026/05/02 02:35:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 02:35:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/02 02:35:54 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmp1rpot_y9\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.1', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


In [25]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)

Training Accuracy: 0.5327734881361664
Validation Accuracy: 0.5335249042145593
Training Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.61      0.60      5915
           1       0.44      0.34      0.39      5921
           2       0.54      0.64      0.59      5907

    accuracy                           0.53     17743
   macro avg       0.52      0.53      0.52     17743
weighted avg       0.52      0.53      0.52     17743

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.61      0.60      1044
           1       0.44      0.32      0.37      1045
           2       0.54      0.67      0.60      1043

    accuracy                           0.53      3132
   macro avg       0.52      0.53      0.52      3132
weighted avg       0.52      0.53      0.52      3132



### SVC

In [26]:
with mlflow.start_run(run_name='svc_v1', experiment_id=experiment_id):
    params = {
        'C': 1.0,                
        'kernel': 'rbf',        
        'gamma': 'scale',       
        'random_state': 42
    }

    svc = SVC(**params)
    svc.fit(X_train, y_train)

    y_train_pred = svc.predict(X_train)
    y_validation_pred = svc.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH)
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.log_params(params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(svc, 'svc_model')

2026/05/02 02:36:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 02:36:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/02 02:36:38 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmpqfoo11kc\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.1', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


In [27]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)

Training Accuracy: 0.3603111086062109
Validation Accuracy: 0.3623882503192848
Training Classification Report:
               precision    recall  f1-score   support

           0       0.47      0.16      0.24      5915
           1       0.34      0.90      0.50      5921
           2       0.60      0.03      0.05      5907

    accuracy                           0.36     17743
   macro avg       0.47      0.36      0.26     17743
weighted avg       0.47      0.36      0.26     17743

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.49      0.17      0.25      1044
           1       0.34      0.89      0.49      1045
           2       0.61      0.03      0.05      1043

    accuracy                           0.36      3132
   macro avg       0.48      0.36      0.27      3132
weighted avg       0.48      0.36      0.27      3132



### decision tree

In [28]:
with mlflow.start_run(run_name='decision_tree_v1', experiment_id=experiment_id):
    params = {
        'criterion': 'gini',       
        'max_depth': 20,           
        'min_samples_split': 5,    
        'min_samples_leaf': 2,     
        'max_features': None,      
        'random_state': 42
    }

    dtc = DecisionTreeClassifier(**params)
    dtc.fit(X_train, y_train)

    y_train_pred = dtc.predict(X_train)
    y_validation_pred = dtc.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH)
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.log_params(params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(dtc, 'decision_tree_model')

2026/05/02 02:36:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 02:36:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/02 02:36:43 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmpqcn3jqny\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.1', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


In [29]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)

Training Accuracy: 0.8662007552274136
Validation Accuracy: 0.6756066411238825
Training Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.93      0.89      5915
           1       0.82      0.84      0.83      5921
           2       0.93      0.83      0.88      5907

    accuracy                           0.87     17743
   macro avg       0.87      0.87      0.87     17743
weighted avg       0.87      0.87      0.87     17743

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.74      0.72      1044
           1       0.58      0.61      0.59      1045
           2       0.76      0.68      0.72      1043

    accuracy                           0.68      3132
   macro avg       0.68      0.68      0.68      3132
weighted avg       0.68      0.68      0.68      3132



### Random forest

In [30]:
with mlflow.start_run(run_name='random_forest_v1', experiment_id=experiment_id):
    params = {
    'n_estimators': 300,          
    'max_depth': 20,             
    'min_samples_split': 5,      
    'min_samples_leaf': 2,        
    'max_features': 'sqrt',       
    'bootstrap': True,
    'random_state': 42,
    'n_jobs': -1                 
    }

    rfc = RandomForestClassifier(**params)
    rfc.fit(X_train, y_train)
    y_train_pred = rfc.predict(X_train)
    y_validation_pred = rfc.predict(X_validation)

    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH)
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.log_params(params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(rfc, 'random_forest_model')

2026/05/02 02:36:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 02:36:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/02 02:36:49 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmpogmdzsdw\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.1', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


In [31]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)

Training Accuracy: 0.871385898664262
Validation Accuracy: 0.7158365261813537
Training Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.89      0.90      5915
           1       0.82      0.84      0.83      5921
           2       0.89      0.88      0.89      5907

    accuracy                           0.87     17743
   macro avg       0.87      0.87      0.87     17743
weighted avg       0.87      0.87      0.87     17743

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.75      0.76      1044
           1       0.61      0.64      0.63      1045
           2       0.76      0.76      0.76      1043

    accuracy                           0.72      3132
   macro avg       0.72      0.72      0.72      3132
weighted avg       0.72      0.72      0.72      3132



### XGBoost

In [32]:
with mlflow.start_run(run_name='xgboost_v1', experiment_id=experiment_id):
    params = {
        'n_estimators': 300,        
        'max_depth': 6,             
        'learning_rate': 0.1,       
        'subsample': 0.8,           
        'colsample_bytree': 0.8,    
        'gamma': 0,                 
        'reg_alpha': 0,             
        'reg_lambda': 1,            
        'random_state': 42,
        'n_jobs': -1,
    }

    xgb = XGBClassifier(**params)
    xgb.fit(X_train, y_train)

    # Predictions
    y_train_pred = xgb.predict(X_train)
    y_validation_pred = xgb.predict(X_validation)

    # Evaluation
    train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
    val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

    # Save confusion matrices as images
    save_confusion_matrix(train_cm, TRAIN_CONFUSION_MATRIX_PATH)
    save_confusion_matrix(val_cm, VALIDATION_CONFUSION_MATRIX_PATH)

    # Logging
    mlflow.log_params(params)
    mlflow.log_metrics({
        'train_acc': train_acc,
        'val_acc': val_acc
    })

    mlflow.log_text(train_report, 'train_classification_report.txt')
    mlflow.log_text(val_report, 'validation_classification_report.txt')

    mlflow.log_artifact(TRAIN_CONFUSION_MATRIX_PATH)
    mlflow.log_artifact(VALIDATION_CONFUSION_MATRIX_PATH)

    mlflow.sklearn.log_model(xgb, 'xgboost_model')

2026/05/02 02:36:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 02:36:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/02 02:36:56 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\KARIMM~1\AppData\Local\Temp\tmpd4dzr_gh\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.1', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


In [33]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)

Training Accuracy: 0.8531251761257961
Validation Accuracy: 0.7273307790549169
Training Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.87      0.88      5915
           1       0.79      0.82      0.81      5921
           2       0.88      0.87      0.87      5907

    accuracy                           0.85     17743
   macro avg       0.85      0.85      0.85     17743
weighted avg       0.85      0.85      0.85     17743

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.76      0.77      1044
           1       0.62      0.66      0.64      1045
           2       0.78      0.76      0.77      1043

    accuracy                           0.73      3132
   macro avg       0.73      0.73      0.73      3132
weighted avg       0.73      0.73      0.73      3132



# TESTING

In [34]:
# # Load model
# run_id = 'a74aa6166c014c009699b9c81c05918e'
# model_name = 'random_forest_classifier'
# model_uri = f'runs:/{run_id}/{model_name}'
# rfc = mlflow.sklearn.load_model(model_uri=model_uri)

# # Predict
# y_pred = rfc.predict(X_test)
# y_pred = pd.DataFrame(y_pred, columns=['prediction'])

# eval_acc, eval_report, eval_cm = evaluate_model(y_pred, y_test)
# print("Test Accuracy:", eval_acc)
# print("Classification Report:\n", eval_report)
# save_confusion_matrix(eval_cm, "test_confusion_matrix.png")